# Ders 11 - Ajanlar Arası (A2A) Protokolü


## Kurulum


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A Protokolü nedir?

The **Ajanlar Arası (A2A) protokolü** farklı çerçeveler üzerinde oluşturulsalar veya farklı servislerde barındırılsalar bile, yapay zeka ajanlarının iletişim kurmasını, birbirlerini keşfetmesini ve işbirliği yapmasını sağlayan açık bir standarttır.

Key concepts:

- **Keşif** – Ajanlar yeteneklerini tanımlayan bir *Ajan Kartı* yayımlar, böylece diğer ajanların (veya orkestratörlerin) bir görev için doğru uzmana ulaşmasını kolaylaştırır.
- **Mesaj Aktarımı** – Ajanlar ortak bir protokol aracılığıyla yapılandırılmış mesajlar alışverişi yapar, böylece bir ajandan gelen istek iç uygulamasından bağımsız olarak başka bir ajan tarafından anlaşılabilir ve yerine getirilebilir.
- **Görev Yaşam Döngüsü** – A2A, *gönderildi*, *çalışıyor*, *tamamlandı* ve *başarısız* gibi durumları tanımlar, böylece orkestratöre devredilen bir görevin nasıl ilerlediği konusunda tam görünürlük sağlar.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## Uzmanlaşmış Seyahat Acenteleri Oluşturma


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## İş Akışı Yoluyla Çok Ajanlı İşbirliği

Üç ajanı, A2A mesaj iletimini yansıtan sıralı bir iş akışına bağlarız:

1. **CurrencyExchangeAgent** kullanıcı isteğini alır ve döviz rehberliği üretir.
2. **ActivityPlannerAgent** zenginleştirilmiş bağlamı alır ve etkinlik önerileri ekler.
3. **TravelManagerAgent** her iki girdiyi nihai bir seyahat özetine dönüştürür.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Üretimde A2A'yi Anlamak

Üretim ortamında A2A protokolü, güçlü servisler arası senaryoların kilidini açar:

| Capability | Description |
|---|---|
| **Çerçeveler arası birlikte çalışabilirlik** | Bir çerçeveyle oluşturulmuş bir ajan, görevleri başka herhangi bir A2A-uyumlu çerçeveyle oluşturulmuş bir ajana devredebilir ve gerçek kuruluşlar arası birlikte çalışabilirlik sağlar. |
| **Servis sınırları** | Ajanlar ayrı mikroservislerde, bulut bölgelerinde veya hatta farklı kuruluşlarda bulunabilirken yine de sorunsuz şekilde işbirliği yapabilir. |
| **Dinamik keşif** | Bir orkestratör, çalışma zamanında bir Agent Card kayıt defterini sorgulayarak belirli bir alt görev için en uygun uzmana ulaşabilir. |
| **Akış & push bildirimleri** | A2A, gerçek zamanlı ilerleme güncellemeleri için Server-Sent Events (SSE) ve uzun süren görevler için push bildirimlerini destekler. |

Yukarıda oluşturduğumuz iş akışı, bu desenin basitleştirilmiş, süreç içi bir versiyonudur. Gerçek bir
dağıtımda her ajan bir HTTP uç noktası sunar, bir Agent Card yayınlar ve
A2A JSON-RPC protokolü aracılığıyla iletişim kurar.


## Özet

Bu derste şunları öğrendiniz:

1. **A2A protokolünün ne olduğu** — ajanlar arası keşif, mesajlaşma,
   ve görev yönetimi için açık bir standart.
2. **Özel ajanlar nasıl oluşturulur** — bir Döviz Değişim ajanı, bir Etkinlik Planlayıcı ajanı,
   ve bir Seyahat Yöneticisi orkestratörü.
3. **Ajanları bir iş akışına nasıl bağlayacağınız** — `WorkflowBuilder` kullanarak ardışık
   ajanlar arasındaki mesaj iletimini modellemek.
4. **A2A'nın üretimde nasıl çalıştığı** — çerçeveler arası, servisler arası işbirliğini
   dinamik keşif ve akış güncellemeleriyle mümkün kılar.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Sorumluluk Reddi:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba göstersek de, otomatik çevirilerin hatalar veya yanlışlıklar içerebileceğini lütfen unutmayın. Belgenin kendi dilindeki orijinal versiyonu yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek herhangi bir yanlış anlama veya yanlış yorumlamadan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
